# NB-702: Agent Memory Systems

Implement various memory types for AI agents.

## Learning Objectives
- Short-term memory
- Long-term memory (vector store)
- Entity memory
- Hybrid memory systems

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
import numpy as np
import time

print("Libraries imported")

## 1. Short-Term Memory

In [ ]:
class ShortTermMemory:
    def __init__(self, max_messages=20):
        self.messages = []
        self.max_messages = max_messages

    def add(self, role: str, content: str):
        """Add a message."""
        self.messages.append({
            "role": role,
            "content": content,
            "timestamp": time.time()
        })

        # Prune old messages
        if len(self.messages) > self.max_messages:
            self.messages = self.messages[-self.max_messages:]

    def get_recent(self, n=5) -> list[dict]:
        """Get n most recent messages."""
        return self.messages[-n:]

    def get_history(self) -> str:
        """Get formatted history."""
        return "\n".join([
            f"{m['role']}: {m['content']}"
            for m in self.messages
        ])

# Test
stm = ShortTermMemory()
stm.add("user", "Hello, I'm Alice")
stm.add("assistant", "Hi Alice!")
print(stm.get_history())

## 2. Long-Term Memory (Vector Store)

In [ ]:
class LongTermMemory:
    def __init__(self, embedder_model="all-MiniLM-L6-v2"):
        self.embedder = SentenceTransformer(embedder_model)
        self.memories = []
        self.embeddings = None

    def add(self, content: str, metadata: dict | None = None):
        """Add a memory."""
        embedding = self.embedder.encode([content])[0]

        self.memories.append({
            "content": content,
            "metadata": metadata or {},
            "timestamp": time.time()
        })

        # Update embeddings
        if self.embeddings is None:
            self.embeddings = embedding.reshape(1, -1)
        else:
            self.embeddings = np.vstack([self.embeddings, embedding])

    def retrieve(self, query: str, top_k=3) -> list[dict]:
        """Retrieve relevant memories."""
        if self.embeddings is None:
            return []

        query_emb = self.embedder.encode([query])[0]
        scores = np.dot(self.embeddings, query_emb)

        top_indices = scores.argsort()[-top_k:][::-1]

        return [self.memories[i] for i in top_indices]

# Test
ltm = LongTermMemory()
ltm.add("Alice loves hiking", {"user": "Alice", "topic": "hobbies"})
ltm.add("Alice is a software engineer", {"user": "Alice", "topic": "work"})

results = ltm.retrieve("What does Alice like?")
for r in results:
    print(f"{r['content']}")

## 3. Entity Memory

In [ ]:
class EntityMemory:
    def __init__(self):
        self.entities: dict[str, dict] = {}

    def extract_and_store(self, text: str):
        """Extract entities from text (simplified)."""

        # TODO: Simple pattern matching
        import re

        # Pattern: "X is Y"
        matches = re.finditer(r"(\w+) is (?:a|an) (\w+(?: \w+)?)", text)

        for match in matches:
            entity = match.group(1).capitalize()
            value = match.group(2)

            if entity not in self.entities:
                self.entities[entity] = {"facts": []}

            self.entities[entity]["facts"].append(value)

    def get_entity(self, name: str) -> dict | None:
        """Get entity information."""
        return self.entities.get(name.capitalize())

# Test
em = EntityMemory()
em.extract_and_store("Alice is a software engineer who loves hiking.")
em.extract_and_store("Bob is a teacher.")

print(f"Alice: {em.get_entity('alice')}")

## 4. Hybrid Memory System

In [ ]:
class HybridMemory:
    def __init__(self):
        self.short_term = ShortTermMemory()
        self.long_term = LongTermMemory()
        self.entity_memory = EntityMemory()

    def add(self, role: str, content: str, metadata: dict | None = None):
        """Add to all memory systems."""

        # Short-term
        self.short_term.add(role, content)

        # Long-term (if important)
        if self._is_important(content):
            self.long_term.add(content, metadata)

        # Entity extraction
        if role == "user":
            self.entity_memory.extract_and_store(content)

    def _is_important(self, text: str) -> bool:
        """Check if text is important."""
        keywords = ["remember", "my name", "i work", "favorite"]
        return any(kw in text.lower() for kw in keywords)

    def get_context(self, query: str) -> str:
        """Get relevant context."""

        context_parts = []

        # Recent messages
        recent = self.short_term.get_recent(3)
        if recent:
            context_parts.append("Recent:\n" + "\n".join([m["content"] for m in recent]))

        # Long-term retrieval
        lt_memories = self.long_term.retrieve(query, top_k=2)
        if lt_memories:
            context_parts.append("Memories:\n" + "\n".join([m["content"] for m in lt_memories]))

        return "\n\n".join(context_parts)

# Test
hm = HybridMemory()
hm.add("user", "My name is Alice and I'm a software engineer")
hm.add("assistant", "Nice to meet you Alice!")

context = hm.get_context("What's my name?")
print(context)

## Exercise

1. Add memory summarization
2. Implement memory importance scoring
3. Add persistent storage